In [ ]:
#模型 1：多任务回归模型（推荐）

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from torch.optim.lr_scheduler import PolynomialLR
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder
from scipy.stats import spearmanr
import os
import random
from tqdm import tqdm
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

warnings.filterwarnings("ignore")

# ====================== 随机种子 ======================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

LOCAL_MODEL_DIR = "./hf_models"
MODEL_PATH = os.path.join(LOCAL_MODEL_DIR, "roberta-base")

# ====================== 数据加载 ======================
print("\n[1/6] 加载数据...")
df = pd.read_csv("mental_heath_unbanlanced.csv")
df = df.dropna(subset=["text", "status"])
df["text"] = df["text"].astype(str)
print(f"数据集大小: {len(df)}")

label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df["status"])
print(f"类别: {label_encoder.classes_}")

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df["text"].tolist(),
    df["label_encoded"].tolist(),
    test_size=0.3,
    random_state=42,
    stratify=df["label_encoded"]
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

print(f"训练集: {len(train_texts)}, 验证集: {len(val_texts)}, 测试集: {len(test_texts)}")

# ====================== 回归目标定义 ======================
def create_regression_targets(single_labels):
    """
    创建三维回归目标
    维度：Depression（抑郁）、Suicidal（自杀风险）、Anxiety（焦虑）
    """
    batch_size = len(single_labels)
    dep_target = torch.zeros(batch_size, dtype=torch.float32)
    sui_target = torch.zeros(batch_size, dtype=torch.float32)
    anx_target = torch.zeros(batch_size, dtype=torch.float32)
    
    for i, lab in enumerate(single_labels):
        if lab == 2:   # Normal
            dep_target[i] = 0.10
            sui_target[i] = 0.03
            anx_target[i] = 0.08
        elif lab == 0: # Anxiety
            dep_target[i] = 0.35
            sui_target[i] = 0.15
            anx_target[i] = 0.75
        elif lab == 1: # Depression
            dep_target[i] = 0.80
            sui_target[i] = 0.45
            anx_target[i] = 0.50
        elif lab == 3: # Suicidal
            dep_target[i] = 0.92
            sui_target[i] = 0.95
            anx_target[i] = 0.65
    
    return dep_target, sui_target, anx_target

# ====================== Dataset ======================
class MultiTaskRegressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=96):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        single_label = self.labels[idx]
        dep_target, sui_target, anx_target = create_regression_targets([single_label])
        
        item = {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "dep_target": dep_target.squeeze(0),
            "sui_target": sui_target.squeeze(0),
            "anx_target": anx_target.squeeze(0),
            "single_label": torch.tensor(single_label, dtype=torch.long),
        }
        return item

# ====================== 多任务回归模型 ======================
class MultiTaskRegressionModel(nn.Module):
    """
    多任务回归模型
    同时回归三个心理健康维度
    """
    def __init__(self, model_path):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_path, local_files_only=True)
        hidden = self.encoder.config.hidden_size
        
        # 特征融合
        self.dropout_main = nn.Dropout(0.2)
        self.layer_norm = nn.LayerNorm(hidden)
        self.attention_weights = nn.Linear(hidden, 1)
        
        # 共享特征层
        self.shared_layer1 = nn.Sequential(
            nn.Linear(hidden, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        
        self.shared_layer2 = nn.Sequential(
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.15)
        )
        
        # 任务特定层
        self.dep_specific = nn.Sequential(
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1)
        )
        
        self.sui_specific = nn.Sequential(
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1)
        )
        
        self.anx_specific = nn.Sequential(
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1)
        )
        
        # 回归头
        self.dep_head = nn.Linear(128, 1)
        self.sui_head = nn.Linear(128, 1)
        self.anx_head = nn.Linear(128, 1)

    def forward(self, input_ids, attention_mask, dep_target=None, sui_target=None, anx_target=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        
        # 多层特征融合
        cls_token = last_hidden[:, 0, :]
        attention_logits = self.attention_weights(last_hidden)
        attention_weights = F.softmax(attention_logits, dim=1)
        attention_pooled = torch.sum(last_hidden * attention_weights, dim=1)
        
        mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        sum_hidden = torch.sum(last_hidden * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        mean_pooled = sum_hidden / sum_mask
        
        fused = cls_token + 0.4 * attention_pooled + 0.3 * mean_pooled
        fused = self.layer_norm(fused)
        fused = self.dropout_main(fused)
        
        # 共享特征提取
        shared = self.shared_layer1(fused)
        shared = self.shared_layer2(shared)
        
        # 任务特定处理
        dep_feat = self.dep_specific(shared)
        sui_feat = self.sui_specific(shared)
        anx_feat = self.anx_specific(shared)
        
        # 回归输出
        dep_score = torch.sigmoid(self.dep_head(dep_feat)).squeeze(-1)
        sui_score = torch.sigmoid(self.sui_head(sui_feat)).squeeze(-1)
        anx_score = torch.sigmoid(self.anx_head(anx_feat)).squeeze(-1)
        
        if dep_target is not None:
            loss_dep = F.mse_loss(dep_score, dep_target)
            loss_sui = F.mse_loss(sui_score, sui_target)
            loss_anx = F.mse_loss(anx_score, anx_target)
            
            total_loss = 0.8 * loss_dep + 1.4 * loss_sui + 0.5 * loss_anx
            
            return {
                "loss": total_loss,
                "loss_dep": loss_dep,
                "loss_sui": loss_sui,
                "loss_anx": loss_anx,
                "dep_score": dep_score,
                "sui_score": sui_score,
                "anx_score": anx_score
            }
        
        return {
            "dep_score": dep_score,
            "sui_score": sui_score,
            "anx_score": anx_score
        }

# ====================== 评估指标 ======================
def concordance_correlation_coefficient(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true)
    var_pred = np.var(y_pred)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    ccc = (2 * cov) / (var_true + var_pred + (mean_true - mean_pred) ** 2 + 1e-10)
    return ccc

# ====================== 训练函数 ======================
def train():
    print("\n[2/6] 加载预训练模型和分词器...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
    
    print("[3/6] 创建数据加载器...")
    train_dataset = MultiTaskRegressionDataset(train_texts, train_labels, tokenizer)
    val_dataset   = MultiTaskRegressionDataset(val_texts,   val_labels,   tokenizer)
    test_dataset  = MultiTaskRegressionDataset(test_texts,  test_labels,  tokenizer)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, drop_last=False)
    val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)
    
    print("[4/6] 初始化模型...")
    model = MultiTaskRegressionModel(MODEL_PATH).to(DEVICE)
    
    optimizer = AdamW(model.parameters(), lr=1.6e-5, weight_decay=1e-5)
    total_steps = len(train_loader) * 8
    scheduler = PolynomialLR(optimizer, total_iters=total_steps, power=1.0)
    
    best_val_loss = float('inf')
    best_state = None
    patience = 4
    patience_counter = 0
    
    # 训练循环
    print("\n[5/6] 开始训练...\n")
    for epoch in range(8):
        model.train()
        total_loss = 0
        total_loss_dep = 0
        total_loss_sui = 0
        total_loss_anx = 0
        
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} Train"):
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            dep_target     = batch["dep_target"].to(DEVICE)
            sui_target     = batch["sui_target"].to(DEVICE)
            anx_target     = batch["anx_target"].to(DEVICE)
            
            outputs = model(input_ids, attention_mask, dep_target, sui_target, anx_target)
            loss = outputs["loss"]
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
            total_loss_dep += outputs["loss_dep"].item()
            total_loss_sui += outputs["loss_sui"].item()
            total_loss_anx += outputs["loss_anx"].item()
        
        avg_train_loss = total_loss / len(train_loader)
        avg_loss_dep = total_loss_dep / len(train_loader)
        avg_loss_sui = total_loss_sui / len(train_loader)
        avg_loss_anx = total_loss_anx / len(train_loader)
        
        print(f"Epoch {epoch+1} Train - Total: {avg_train_loss:.4f} | Dep: {avg_loss_dep:.4f} | Sui: {avg_loss_sui:.4f} | Anx: {avg_loss_anx:.4f}")
        
        # 验证
        model.eval()
        val_dep_preds = []
        val_sui_preds = []
        val_anx_preds = []
        val_dep_targets = []
        val_sui_targets = []
        val_anx_targets = []
        val_labels_list = []
        val_loss = 0
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                dep_target     = batch["dep_target"].to(DEVICE)
                sui_target     = batch["sui_target"].to(DEVICE)
                anx_target     = batch["anx_target"].to(DEVICE)
                single_labels  = batch["single_label"].numpy()
                
                outputs = model(input_ids, attention_mask, dep_target, sui_target, anx_target)
                val_loss += outputs["loss"].item()
                
                val_dep_preds.extend(outputs["dep_score"].cpu().numpy())
                val_sui_preds.extend(outputs["sui_score"].cpu().numpy())
                val_anx_preds.extend(outputs["anx_score"].cpu().numpy())
                val_dep_targets.extend(dep_target.cpu().numpy())
                val_sui_targets.extend(sui_target.cpu().numpy())
                val_anx_targets.extend(anx_target.cpu().numpy())
                val_labels_list.extend(single_labels)
        
        avg_val_loss = val_loss / len(val_loader)
        val_dep_spear, _ = spearmanr(val_dep_targets, val_dep_preds)
        val_sui_spear, _ = spearmanr(val_sui_targets, val_sui_preds)
        val_anx_spear, _ = spearmanr(val_anx_targets, val_anx_preds)
        
        print(f"Epoch {epoch+1} Val - Loss: {avg_val_loss:.4f} | Dep: {val_dep_spear:.4f} | Sui: {val_sui_spear:.4f} | Anx: {val_anx_spear:.4f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_state = model.state_dict()
            torch.save(best_state, "best_multi_task_regression.pt")
            patience_counter = 0
            print(f"✓ Best model saved (Val Loss: {avg_val_loss:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    # 测试
    print("\n[6/6] 开始测试评估...\n")
    model.load_state_dict(torch.load("best_multi_task_regression.pt"))
    model.eval()
    
    test_dep_preds = []
    test_sui_preds = []
    test_anx_preds = []
    test_dep_targets = []
    test_sui_targets = []
    test_anx_targets = []
    test_labels_list = []
    
    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            dep_target     = batch["dep_target"].cpu().numpy()
            sui_target     = batch["sui_target"].cpu().numpy()
            anx_target     = batch["anx_target"].cpu().numpy()
            single_labels  = batch["single_label"].numpy()
            
            outputs = model(input_ids, attention_mask)
            
            test_dep_preds.extend(outputs["dep_score"].cpu().numpy())
            test_sui_preds.extend(outputs["sui_score"].cpu().numpy())
            test_anx_preds.extend(outputs["anx_score"].cpu().numpy())
            test_dep_targets.extend(dep_target)
            test_sui_targets.extend(sui_target)
            test_anx_targets.extend(anx_target)
            test_labels_list.extend(single_labels)
    
    # 转换为数组
    test_dep_preds = np.array(test_dep_preds)
    test_sui_preds = np.array(test_sui_preds)
    test_anx_preds = np.array(test_anx_preds)
    test_dep_targets = np.array(test_dep_targets)
    test_sui_targets = np.array(test_sui_targets)
    test_anx_targets = np.array(test_anx_targets)
    test_labels_list = np.array(test_labels_list)
    
    # 计算指标
    dep_spear, dep_pvalue = spearmanr(test_dep_targets, test_dep_preds)
    dep_ccc = concordance_correlation_coefficient(test_dep_targets, test_dep_preds)
    dep_mae = np.mean(np.abs(test_dep_targets - test_dep_preds))
    dep_rmse = np.sqrt(np.mean((test_dep_targets - test_dep_preds) ** 2))
    
    sui_spear, sui_pvalue = spearmanr(test_sui_targets, test_sui_preds)
    sui_ccc = concordance_correlation_coefficient(test_sui_targets, test_sui_preds)
    sui_mae = np.mean(np.abs(test_sui_targets - test_sui_preds))
    sui_rmse = np.sqrt(np.mean((test_sui_targets - test_sui_preds) ** 2))
    
    anx_spear, anx_pvalue = spearmanr(test_anx_targets, test_anx_preds)
    anx_ccc = concordance_correlation_coefficient(test_anx_targets, test_anx_preds)
    anx_mae = np.mean(np.abs(test_anx_targets - test_anx_preds))
    anx_rmse = np.sqrt(np.mean((test_anx_targets - test_anx_preds) ** 2))
    
    # 打印结果
    print("\n" + "="*80)
    print("=== 多任务回归模型 - 测试结果 ===")
    print("="*80)
    print(f"\nDepression Risk:")
    print(f"  Spearman: {dep_spear:.4f} (p-value: {dep_pvalue:.2e})")
    print(f"  CCC: {dep_ccc:.4f}")
    print(f"  MAE: {dep_mae:.4f}")
    print(f"  RMSE: {dep_rmse:.4f}")
    
    print(f"\nSuicidal Risk:")
    print(f"  Spearman: {sui_spear:.4f} (p-value: {sui_pvalue:.2e})")
    print(f"  CCC: {sui_ccc:.4f}")
    print(f"  MAE: {sui_mae:.4f}")
    print(f"  RMSE: {sui_rmse:.4f}")
    
    print(f"\nAnxiety:")
    print(f"  Spearman: {anx_spear:.4f} (p-value: {anx_pvalue:.2e})")
    print(f"  CCC: {anx_ccc:.4f}")
    print(f"  MAE: {anx_mae:.4f}")
    print(f"  RMSE: {anx_rmse:.4f}")
    
    avg_spear = (dep_spear + sui_spear + anx_spear) / 3
    avg_ccc = (dep_ccc + sui_ccc + anx_ccc) / 3
    print(f"\n平均 Spearman: {avg_spear:.4f}")
    print(f"平均 CCC: {avg_ccc:.4f}")
    
    # 高风险检测
    high_risk_true = (test_labels_list == 3).astype(int)
    threshold = 0.72
    high_risk_pred = (test_sui_preds >= threshold).astype(int)
    
    auc = roc_auc_score(high_risk_true, test_sui_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(high_risk_true, high_risk_pred, average='binary')
    
    print("\n" + "="*80)
    print(f"=== 高风险检测 (threshold = {threshold}) ===")
    print("="*80)
    print(f"AUC: {auc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    
    # 可视化
    print("\n生成可视化图表...")
    
    # 2D Risk Space
    plt.figure(figsize=(12, 9))
    colors = ['#3498db', '#e74c3c', '#95a5a6', '#c0392b']
    class_names = label_encoder.classes_
    
    for i in range(4):
        mask = test_labels_list == i
        plt.scatter(test_dep_preds[mask], 
                    test_sui_preds[mask], 
                    c=colors[i], label=f'{class_names[i]} (n={mask.sum()})', 
                    alpha=0.7, s=50, edgecolors='black', linewidth=0.5)
    
    plt.axvline(x=0.60, color='orange', linestyle='--', alpha=0.5, linewidth=2)
    plt.axhline(y=threshold, color='red', linestyle='--', alpha=0.5, linewidth=2)
    
    plt.xlabel('Predicted Depression Severity', fontsize=12, fontweight='bold')
    plt.ylabel('Predicted Suicidal Risk', fontsize=12, fontweight='bold')
    plt.title('2D Risk Space - Multi-Task Regression', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10, loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.xlim(-0.05, 1.05)
    plt.ylim(-0.05, 1.05)
    plt.tight_layout()
    plt.savefig('1_2d_risk_space.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ 2D risk space 已保存为 1_2d_risk_space.png")
    
    # 预测 vs 真实值
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].scatter(test_dep_targets, test_dep_preds, alpha=0.6, s=30, c='#3498db', edgecolors='black', linewidth=0.5)
    axes[0].plot([0, 1], [0, 1], 'r--', lw=2)
    axes[0].set_xlabel('True Target', fontweight='bold')
    axes[0].set_ylabel('Predicted Score', fontweight='bold')
    axes[0].set_title(f'Depression\nSpearman={dep_spear:.4f}, CCC={dep_ccc:.4f}', fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim(-0.05, 1.05)
    axes[0].set_ylim(-0.05, 1.05)
    
    axes[1].scatter(test_sui_targets, test_sui_preds, alpha=0.6, s=30, c='#e74c3c', edgecolors='black', linewidth=0.5)
    axes[1].plot([0, 1], [0, 1], 'r--', lw=2)
    axes[1].set_xlabel('True Target', fontweight='bold')
    axes[1].set_ylabel('Predicted Score', fontweight='bold')
    axes[1].set_title(f'Suicidal Risk\nSpearman={sui_spear:.4f}, CCC={sui_ccc:.4f}', fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim(-0.05, 1.05)
    axes[1].set_ylim(-0.05, 1.05)
    
    axes[2].scatter(test_anx_targets, test_anx_preds, alpha=0.6, s=30, c='#f39c12', edgecolors='black', linewidth=0.5)
    axes[2].plot([0, 1], [0, 1], 'r--', lw=2)
    axes[2].set_xlabel('True Target', fontweight='bold')
    axes[2].set_ylabel('Predicted Score', fontweight='bold')
    axes[2].set_title(f'Anxiety\nSpearman={anx_spear:.4f}, CCC={anx_ccc:.4f}', fontweight='bold')
    axes[2].grid(True, alpha=0.3)
    axes[2].set_xlim(-0.05, 1.05)
    axes[2].set_ylim(-0.05, 1.05)
    
    plt.tight_layout()
    plt.savefig('1_prediction_vs_target.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ 预测 vs 真实值已保存为 1_prediction_vs_target.png")
    
    # 分布图
    df_violin = pd.DataFrame({
        'Depression Score': test_dep_preds,
        'Suicidal Risk Score': test_sui_preds,
        'Anxiety Score': test_anx_preds,
        'True Class': [class_names[l] for l in test_labels_list]
    })
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    sns.violinplot(data=df_violin, x='True Class', y='Depression Score', ax=axes[0], palette='Set2')
    axes[0].set_title('Depression Score Distribution', fontweight='bold')
    axes[0].set_ylabel('Predicted Score (0-1)')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    sns.violinplot(data=df_violin, x='True Class', y='Suicidal Risk Score', ax=axes[1], palette='Set2')
    axes[1].set_title('Suicidal Risk Score Distribution', fontweight='bold')
    axes[1].set_ylabel('Predicted Score (0-1)')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    sns.violinplot(data=df_violin, x='True Class', y='Anxiety Score', ax=axes[2], palette='Set2')
    axes[2].set_title('Anxiety Score Distribution', fontweight='bold')
    axes[2].set_ylabel('Predicted Score (0-1)')
    axes[2].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('1_score_distribution.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ 分布图已保存为 1_score_distribution.png")
    
    print("\n✓ 多任务回归模型实验完成！\n")

if __name__ == "__main__":
    train()


Using device: cuda

[1/6] 加载数据...
数据集大小: 49612
类别: ['Anxiety' 'Depression' 'Normal' 'Suicidal']
训练集: 34728, 验证集: 7442, 测试集: 7442

[2/6] 加载预训练模型和分词器...
[3/6] 创建数据加载器...
[4/6] 初始化模型...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: ./hf_models\roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



[5/6] 开始训练...



Epoch 1 Train: 100%|█████████████████████████████████████████████████████████████| 2171/2171 [15:55<00:00,  2.27it/s]


...

Epoch 6 Train - Total: 0.0201 | Dep: 0.0044 | Sui: 0.0106 | Anx: 0.0036
Epoch 6 Val - Loss: 0.0545 | Dep: 0.9071 | Sui: 0.9070 | Anx: 0.8806


Epoch 7 Train: 100%|█████████████████████████████████████████████████████████████| 2171/2171 [16:14<00:00,  2.23it/s]


Epoch 7 Train - Total: 0.0155 | Dep: 0.0033 | Sui: 0.0082 | Anx: 0.0028
Epoch 7 Val - Loss: 0.0572 | Dep: 0.9078 | Sui: 0.9067 | Anx: 0.8800


Epoch 8 Train: 100%|█████████████████████████████████████████████████████████████| 2171/2171 [16:18<00:00,  2.22it/s]


Epoch 8 Train - Total: 0.0128 | Dep: 0.0027 | Sui: 0.0067 | Anx: 0.0024
Epoch 8 Val - Loss: 0.0581 | Dep: 0.9070 | Sui: 0.9056 | Anx: 0.8785
Early stopping at epoch 8

[6/6] 开始测试评估...


=== 多任务回归模型 - 测试结果 ===

Depression Risk:
  Spearman: 0.9014 (p-value: 0.00e+00)
  CCC: 0.9300
  MAE: 0.0599
  RMSE: 0.1340

Suicidal Risk:
  Spearman: 0.8993 (p-value: 0.00e+00)
  CCC: 0.8898
  MAE: 0.0886
  RMSE: 0.1634

Anxiety:
  Spearman: 0.8644 (p-value: 0.00e+00)
  CCC: 0.9104
  MAE: 0.0544
  RMSE: 0.1112

平均 Spearman: 0.8884
平均 CCC: 0.9101

=== 高风险检测 (threshold = 0.72) ===
AUC: 0.9525
Precision: 0.7520
Recall: 0.8401
F1-Score: 0.7936

生成可视化图表...
✓ 2D risk space 已保存为 1_2d_risk_space.png
✓ 预测 vs 真实值已保存为 1_prediction_vs_target.png
✓ 分布图已保存为 1_score_distribution.png

✓ 多任务回归模型实验完成！



In [ ]:
#模型 2：对偶回归模型（Dual Regression）

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from torch.optim.lr_scheduler import PolynomialLR
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder
from scipy.stats import spearmanr
import os
import random
from tqdm import tqdm
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

LOCAL_MODEL_DIR = "./hf_models"
MODEL_PATH = os.path.join(LOCAL_MODEL_DIR, "roberta-base")

# ====================== 数据加载 ======================
print("\n[1/6] 加载数据...")
df = pd.read_csv("mental_heath_unbanlanced.csv")
df = df.dropna(subset=["text", "status"])
df["text"] = df["text"].astype(str)
print(f"数据集大小: {len(df)}")

label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df["status"])
print(f"类别: {label_encoder.classes_}")

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df["text"].tolist(),
    df["label_encoded"].tolist(),
    test_size=0.3,
    random_state=42,
    stratify=df["label_encoded"]
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

print(f"训练集: {len(train_texts)}, 验证集: {len(val_texts)}, 测试集: {len(test_texts)}")

# ====================== 回归目标定义 ======================
def create_regression_targets(single_labels):
    """
    对偶回归：主维度（Depression）和次维度（Suicidal）
    使用相关性约束
    """
    batch_size = len(single_labels)
    dep_target = torch.zeros(batch_size, dtype=torch.float32)
    sui_target = torch.zeros(batch_size, dtype=torch.float32)
    
    for i, lab in enumerate(single_labels):
        if lab == 2:   # Normal
            dep_target[i] = 0.10
            sui_target[i] = 0.03
        elif lab == 0: # Anxiety
            dep_target[i] = 0.35
            sui_target[i] = 0.15
        elif lab == 1: # Depression
            dep_target[i] = 0.80
            sui_target[i] = 0.45
        elif lab == 3: # Suicidal
            dep_target[i] = 0.92
            sui_target[i] = 0.95
    
    return dep_target, sui_target

# ====================== Dataset ======================
class DualRegressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=96):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        single_label = self.labels[idx]
        dep_target, sui_target = create_regression_targets([single_label])
        
        item = {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "dep_target": dep_target.squeeze(0),
            "sui_target": sui_target.squeeze(0),
            "single_label": torch.tensor(single_label, dtype=torch.long),
        }
        return item

# ====================== 对偶回归模型 ======================
class DualRegressionModel(nn.Module):
    """
    对偶回归模型
    两个回归头共享深层特征，但有独立的中层表示
    使用相关性约束来建立两个维度之间的联系
    """
    def __init__(self, model_path):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_path, local_files_only=True)
        hidden = self.encoder.config.hidden_size
        
        # 特征融合
        self.dropout_main = nn.Dropout(0.2)
        self.layer_norm = nn.LayerNorm(hidden)
        self.attention_weights = nn.Linear(hidden, 1)
        
        # 共享深层特征
        self.shared_deep = nn.Sequential(
            nn.Linear(hidden, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        
        # Depression 特定的中层
        self.dep_middle = nn.Sequential(
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.15)
        )
        
        # Suicidal 特定的中层
        self.sui_middle = nn.Sequential(
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.15)
        )
        
        # 共享浅层特征（用于相关性约束）
        self.shared_shallow = nn.Sequential(
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1)
        )
        
        # 回归头
        self.dep_head = nn.Linear(128, 1)
        self.sui_head = nn.Linear(128, 1)

    def forward(self, input_ids, attention_mask, dep_target=None, sui_target=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        
        # 特征融合
        cls_token = last_hidden[:, 0, :]
        attention_logits = self.attention_weights(last_hidden)
        attention_weights = F.softmax(attention_logits, dim=1)
        attention_pooled = torch.sum(last_hidden * attention_weights, dim=1)
        
        mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        sum_hidden = torch.sum(last_hidden * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        mean_pooled = sum_hidden / sum_mask
        
        fused = cls_token + 0.4 * attention_pooled + 0.3 * mean_pooled
        fused = self.layer_norm(fused)
        fused = self.dropout_main(fused)
        
        # 共享深层
        shared_deep = self.shared_deep(fused)
        
        # 对偶路径
        dep_feat = self.dep_middle(shared_deep)
        sui_feat = self.sui_middle(shared_deep)
        
        # 共享浅层（融合两个路径）
        dep_feat_shallow = self.shared_shallow(dep_feat)
        sui_feat_shallow = self.shared_shallow(sui_feat)
        
        # 融合两个特征
        fused_feat = (dep_feat_shallow + sui_feat_shallow) / 2
        
        # 回归输出
        dep_score = torch.sigmoid(self.dep_head(fused_feat)).squeeze(-1)
        sui_score = torch.sigmoid(self.sui_head(fused_feat)).squeeze(-1)
        
        if dep_target is not None:
            loss_dep = F.mse_loss(dep_score, dep_target)
            loss_sui = F.mse_loss(sui_score, sui_target)
            
            # 相关性约束：两个维度应该有正相关
            correlation_loss = F.mse_loss(dep_score, sui_score * 0.95)
            
            total_loss = 0.8 * loss_dep + 1.4 * loss_sui + 0.1 * correlation_loss
            
            return {
                "loss": total_loss,
                "loss_dep": loss_dep,
                "loss_sui": loss_sui,
                "dep_score": dep_score,
                "sui_score": sui_score
            }
        
        return {
            "dep_score": dep_score,
            "sui_score": sui_score
        }

# ====================== 评估指标 ======================
def concordance_correlation_coefficient(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true)
    var_pred = np.var(y_pred)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    ccc = (2 * cov) / (var_true + var_pred + (mean_true - mean_pred) ** 2 + 1e-10)
    return ccc

# ====================== 训练函数 ======================
def train():
    print("\n[2/6] 加载预训练模型和分词器...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
    
    print("[3/6] 创建数据加载器...")
    train_dataset = DualRegressionDataset(train_texts, train_labels, tokenizer)
    val_dataset   = DualRegressionDataset(val_texts,   val_labels,   tokenizer)
    test_dataset  = DualRegressionDataset(test_texts,  test_labels,  tokenizer)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, drop_last=False)
    val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)
    
    print("[4/6] 初始化模型...")
    model = DualRegressionModel(MODEL_PATH).to(DEVICE)
    
    optimizer = AdamW(model.parameters(), lr=1.6e-5, weight_decay=1e-5)
    total_steps = len(train_loader) * 8
    scheduler = PolynomialLR(optimizer, total_iters=total_steps, power=1.0)
    
    best_val_loss = float('inf')
    best_state = None
    patience = 4
    patience_counter = 0
    
    print("\n[5/6] 开始训练...\n")
    for epoch in range(8):
        model.train()
        total_loss = 0
        total_loss_dep = 0
        total_loss_sui = 0
        
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} Train"):
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            dep_target     = batch["dep_target"].to(DEVICE)
            sui_target     = batch["sui_target"].to(DEVICE)
            
            outputs = model(input_ids, attention_mask, dep_target, sui_target)
            loss = outputs["loss"]
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
            total_loss_dep += outputs["loss_dep"].item()
            total_loss_sui += outputs["loss_sui"].item()
        
        avg_train_loss = total_loss / len(train_loader)
        avg_loss_dep = total_loss_dep / len(train_loader)
        avg_loss_sui = total_loss_sui / len(train_loader)
        
        print(f"Epoch {epoch+1} Train - Total: {avg_train_loss:.4f} | Dep: {avg_loss_dep:.4f} | Sui: {avg_loss_sui:.4f}")
        
        # 验证
        model.eval()
        val_dep_preds = []
        val_sui_preds = []
        val_dep_targets = []
        val_sui_targets = []
        val_labels_list = []
        val_loss = 0
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                dep_target     = batch["dep_target"].to(DEVICE)
                sui_target     = batch["sui_target"].to(DEVICE)
                single_labels  = batch["single_label"].numpy()
                
                outputs = model(input_ids, attention_mask, dep_target, sui_target)
                val_loss += outputs["loss"].item()
                
                val_dep_preds.extend(outputs["dep_score"].cpu().numpy())
                val_sui_preds.extend(outputs["sui_score"].cpu().numpy())
                val_dep_targets.extend(dep_target.cpu().numpy())
                val_sui_targets.extend(sui_target.cpu().numpy())
                val_labels_list.extend(single_labels)
        
        avg_val_loss = val_loss / len(val_loader)
        val_dep_spear, _ = spearmanr(val_dep_targets, val_dep_preds)
        val_sui_spear, _ = spearmanr(val_sui_targets, val_sui_preds)
        
        print(f"Epoch {epoch+1} Val - Loss: {avg_val_loss:.4f} | Dep: {val_dep_spear:.4f} | Sui: {val_sui_spear:.4f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_state = model.state_dict()
            torch.save(best_state, "best_dual_regression.pt")
            patience_counter = 0
            print(f"✓ Best model saved (Val Loss: {avg_val_loss:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    # 测试
    print("\n[6/6] 开始测试评估...\n")
    model.load_state_dict(torch.load("best_dual_regression.pt"))
    model.eval()
    
    test_dep_preds = []
    test_sui_preds = []
    test_dep_targets = []
    test_sui_targets = []
    test_labels_list = []
    
    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            dep_target     = batch["dep_target"].cpu().numpy()
            sui_target     = batch["sui_target"].cpu().numpy()
            single_labels  = batch["single_label"].numpy()
            
            outputs = model(input_ids, attention_mask)
            
            test_dep_preds.extend(outputs["dep_score"].cpu().numpy())
            test_sui_preds.extend(outputs["sui_score"].cpu().numpy())
            test_dep_targets.extend(dep_target)
            test_sui_targets.extend(sui_target)
            test_labels_list.extend(single_labels)
    
    test_dep_preds = np.array(test_dep_preds)
    test_sui_preds = np.array(test_sui_preds)
    test_dep_targets = np.array(test_dep_targets)
    test_sui_targets = np.array(test_sui_targets)
    test_labels_list = np.array(test_labels_list)
    
    # 计算指标
    dep_spear, dep_pvalue = spearmanr(test_dep_targets, test_dep_preds)
    dep_ccc = concordance_correlation_coefficient(test_dep_targets, test_dep_preds)
    dep_mae = np.mean(np.abs(test_dep_targets - test_dep_preds))
    dep_rmse = np.sqrt(np.mean((test_dep_targets - test_dep_preds) ** 2))
    
    sui_spear, sui_pvalue = spearmanr(test_sui_targets, test_sui_preds)
    sui_ccc = concordance_correlation_coefficient(test_sui_targets, test_sui_preds)
    sui_mae = np.mean(np.abs(test_sui_targets - test_sui_preds))
    sui_rmse = np.sqrt(np.mean((test_sui_targets - test_sui_preds) ** 2))
    
    print("\n" + "="*80)
    print("=== 对偶回归模型 - 测试结果 ===")
    print("="*80)
    print(f"\nDepression Risk:")
    print(f"  Spearman: {dep_spear:.4f} (p-value: {dep_pvalue:.2e})")
    print(f"  CCC: {dep_ccc:.4f}")
    print(f"  MAE: {dep_mae:.4f}")
    print(f"  RMSE: {dep_rmse:.4f}")
    
    print(f"\nSuicidal Risk:")
    print(f"  Spearman: {sui_spear:.4f} (p-value: {sui_pvalue:.2e})")
    print(f"  CCC: {sui_ccc:.4f}")
    print(f"  MAE: {sui_mae:.4f}")
    print(f"  RMSE: {sui_rmse:.4f}")
    
    avg_spear = (dep_spear + sui_spear) / 2
    avg_ccc = (dep_ccc + sui_ccc) / 2
    print(f"\n平均 Spearman: {avg_spear:.4f}")
    print(f"平均 CCC: {avg_ccc:.4f}")
    
    # 高风险检测
    high_risk_true = (test_labels_list == 3).astype(int)
    threshold = 0.72
    high_risk_pred = (test_sui_preds >= threshold).astype(int)
    
    auc = roc_auc_score(high_risk_true, test_sui_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(high_risk_true, high_risk_pred, average='binary')
    
    print("\n" + "="*80)
    print(f"=== 高风险检测 (threshold = {threshold}) ===")
    print("="*80)
    print(f"AUC: {auc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    
    # 可视化
    print("\n生成可视化图表...")
    
    plt.figure(figsize=(12, 9))
    colors = ['#3498db', '#e74c3c', '#95a5a6', '#c0392b']
    class_names = label_encoder.classes_
    
    for i in range(4):
        mask = test_labels_list == i
        plt.scatter(test_dep_preds[mask], 
                    test_sui_preds[mask], 
                    c=colors[i], label=f'{class_names[i]} (n={mask.sum()})', 
                    alpha=0.7, s=50, edgecolors='black', linewidth=0.5)
    
    plt.axvline(x=0.60, color='orange', linestyle='--', alpha=0.5, linewidth=2)
    plt.axhline(y=threshold, color='red', linestyle='--', alpha=0.5, linewidth=2)
    
    plt.xlabel('Predicted Depression Severity', fontsize=12, fontweight='bold')
    plt.ylabel('Predicted Suicidal Risk', fontsize=12, fontweight='bold')
    plt.title('2D Risk Space - Dual Regression', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10, loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.xlim(-0.05, 1.05)
    plt.ylim(-0.05, 1.05)
    plt.tight_layout()
    plt.savefig('2_2d_risk_space.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ 2D risk space 已保存为 2_2d_risk_space.png")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].scatter(test_dep_targets, test_dep_preds, alpha=0.6, s=30, c='#3498db', edgecolors='black', linewidth=0.5)
    axes[0].plot([0, 1], [0, 1], 'r--', lw=2)
    axes[0].set_xlabel('True Target', fontweight='bold')
    axes[0].set_ylabel('Predicted Score', fontweight='bold')
    axes[0].set_title(f'Depression\nSpearman={dep_spear:.4f}, CCC={dep_ccc:.4f}', fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim(-0.05, 1.05)
    axes[0].set_ylim(-0.05, 1.05)
    
    axes[1].scatter(test_sui_targets, test_sui_preds, alpha=0.6, s=30, c='#e74c3c', edgecolors='black', linewidth=0.5)
    axes[1].plot([0, 1], [0, 1], 'r--', lw=2)
    axes[1].set_xlabel('True Target', fontweight='bold')
    axes[1].set_ylabel('Predicted Score', fontweight='bold')
    axes[1].set_title(f'Suicidal Risk\nSpearman={sui_spear:.4f}, CCC={sui_ccc:.4f}', fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim(-0.05, 1.05)
    axes[1].set_ylim(-0.05, 1.05)
    
    plt.tight_layout()
    plt.savefig('2_prediction_vs_target.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ 预测 vs 真实值已保存为 2_prediction_vs_target.png")
    
    df_violin = pd.DataFrame({
        'Depression Score': test_dep_preds,
        'Suicidal Risk Score': test_sui_preds,
        'True Class': [class_names[l] for l in test_labels_list]
    })
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    sns.violinplot(data=df_violin, x='True Class', y='Depression Score', ax=axes[0], palette='Set2')
    axes[0].set_title('Depression Score Distribution', fontweight='bold')
    axes[0].set_ylabel('Predicted Score (0-1)')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    sns.violinplot(data=df_violin, x='True Class', y='Suicidal Risk Score', ax=axes[1], palette='Set2')
    axes[1].set_title('Suicidal Risk Score Distribution', fontweight='bold')
    axes[1].set_ylabel('Predicted Score (0-1)')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('2_score_distribution.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ 分布图已保存为 2_score_distribution.png")
    
    print("\n✓ 对偶回归模型实验完成！\n")

if __name__ == "__main__":
    train()


Using device: cuda

[1/6] 加载数据...
数据集大小: 49612
类别: ['Anxiety' 'Depression' 'Normal' 'Suicidal']
训练集: 34728, 验证集: 7442, 测试集: 7442

[2/6] 加载预训练模型和分词器...
[3/6] 创建数据加载器...
[4/6] 初始化模型...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: ./hf_models\roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



[5/6] 开始训练...



Epoch 1 Train: 100%|█████████████████████████████████████████████████████████████| 2171/2171 [16:36<00:00,  2.18it/s]


...

Epoch 4 Train - Total: 0.0326 | Dep: 0.0082 | Sui: 0.0165
Epoch 4 Val - Loss: 0.0532 | Dep: 0.9055 | Sui: 0.9043


Epoch 5 Train: 100%|█████████████████████████████████████████████████████████████| 2171/2171 [16:02<00:00,  2.26it/s]


Epoch 5 Train - Total: 0.0270 | Dep: 0.0065 | Sui: 0.0134
Epoch 5 Val - Loss: 0.0534 | Dep: 0.9064 | Sui: 0.9062


Epoch 6 Train: 100%|█████████████████████████████████████████████████████████████| 2171/2171 [08:39<00:00,  4.18it/s]


Epoch 6 Train - Total: 0.0220 | Dep: 0.0054 | Sui: 0.0104
Epoch 6 Val - Loss: 0.0552 | Dep: 0.9064 | Sui: 0.9062
Early stopping at epoch 6

[6/6] 开始测试评估...


=== 对偶回归模型 - 测试结果 ===

Depression Risk:
  Spearman: 0.9068 (p-value: 0.00e+00)
  CCC: 0.9362
  MAE: 0.0573
  RMSE: 0.1288

Suicidal Risk:
  Spearman: 0.9059 (p-value: 0.00e+00)
  CCC: 0.8931
  MAE: 0.0848
  RMSE: 0.1616

平均 Spearman: 0.9063
平均 CCC: 0.9147

=== 高风险检测 (threshold = 0.72) ===
AUC: 0.9520
Precision: 0.7635
Recall: 0.8157
F1-Score: 0.7887

生成可视化图表...
✓ 2D risk space 已保存为 2_2d_risk_space.png
✓ 预测 vs 真实值已保存为 2_prediction_vs_target.png
✓ 分布图已保存为 2_score_distribution.png

✓ 对偶回归模型实验完成！



In [3]:
#模型 3：联合回归模型（Joint Regression）

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from torch.optim.lr_scheduler import PolynomialLR
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder
from scipy.stats import spearmanr
import os
import random
from tqdm import tqdm
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

LOCAL_MODEL_DIR = "./hf_models"
MODEL_PATH = os.path.join(LOCAL_MODEL_DIR, "roberta-base")

# ====================== 数据加载 ======================
print("\n[1/6] 加载数据...")
df = pd.read_csv("mental_heath_unbanlanced.csv")
df = df.dropna(subset=["text", "status"])
df["text"] = df["text"].astype(str)
print(f"数据集大小: {len(df)}")

label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df["status"])
print(f"类别: {label_encoder.classes_}")

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df["text"].tolist(),
    df["label_encoded"].tolist(),
    test_size=0.3,
    random_state=42,
    stratify=df["label_encoded"]
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

print(f"训练集: {len(train_texts)}, 验证集: {len(val_texts)}, 测试集: {len(test_texts)}")

# ====================== 回归目标定义 ======================
def create_regression_targets(single_labels):
    """
    联合回归：创建综合风险评分
    结合多个维度的信息
    """
    batch_size = len(single_labels)
    dep_target = torch.zeros(batch_size, dtype=torch.float32)
    sui_target = torch.zeros(batch_size, dtype=torch.float32)
    risk_target = torch.zeros(batch_size, dtype=torch.float32)  # 综合风险
    
    for i, lab in enumerate(single_labels):
        if lab == 2:   # Normal
            dep_target[i] = 0.10
            sui_target[i] = 0.03
            risk_target[i] = 0.05
        elif lab == 0: # Anxiety
            dep_target[i] = 0.35
            sui_target[i] = 0.15
            risk_target[i] = 0.30
        elif lab == 1: # Depression
            dep_target[i] = 0.80
            sui_target[i] = 0.45
            risk_target[i] = 0.70
        elif lab == 3: # Suicidal
            dep_target[i] = 0.92
            sui_target[i] = 0.95
            risk_target[i] = 0.95
    
    return dep_target, sui_target, risk_target

# ====================== Dataset ======================
class JointRegressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=96):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        single_label = self.labels[idx]
        dep_target, sui_target, risk_target = create_regression_targets([single_label])
        
        item = {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "dep_target": dep_target.squeeze(0),
            "sui_target": sui_target.squeeze(0),
            "risk_target": risk_target.squeeze(0),
            "single_label": torch.tensor(single_label, dtype=torch.long),
        }
        return item

# ====================== 联合回归模型 ======================
class JointRegressionModel(nn.Module):
    """
    联合回归模型
    同时预测单个维度和综合风险评分
    使用联合学习框架
    """
    def __init__(self, model_path):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_path, local_files_only=True)
        hidden = self.encoder.config.hidden_size
        
        # 特征融合
        self.dropout_main = nn.Dropout(0.2)
        self.layer_norm = nn.LayerNorm(hidden)
        self.attention_weights = nn.Linear(hidden, 1)
        
        # 共享特征提取
        self.shared_layer1 = nn.Sequential(
            nn.Linear(hidden, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        
        self.shared_layer2 = nn.Sequential(
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.15)
        )
        
        # 单个维度的特定层
        self.dep_specific = nn.Sequential(
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1)
        )
        
        self.sui_specific = nn.Sequential(
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1)
        )
        
        # 综合风险的特定层
        self.risk_specific = nn.Sequential(
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1)
        )
        
        # 回归头
        self.dep_head = nn.Linear(128, 1)
        self.sui_head = nn.Linear(128, 1)
        self.risk_head = nn.Linear(128, 1)
        
        # 融合层（用于联合学习）
        self.fusion_layer = nn.Sequential(
            nn.Linear(128 * 3, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.1)
        )
        
        self.fusion_head = nn.Linear(256, 1)

    def forward(self, input_ids, attention_mask, dep_target=None, sui_target=None, risk_target=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        
        # 特征融合
        cls_token = last_hidden[:, 0, :]
        attention_logits = self.attention_weights(last_hidden)
        attention_weights = F.softmax(attention_logits, dim=1)
        attention_pooled = torch.sum(last_hidden * attention_weights, dim=1)
        
        mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        sum_hidden = torch.sum(last_hidden * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        mean_pooled = sum_hidden / sum_mask
        
        fused = cls_token + 0.4 * attention_pooled + 0.3 * mean_pooled
        fused = self.layer_norm(fused)
        fused = self.dropout_main(fused)
        
        # 共享特征提取
        shared = self.shared_layer1(fused)
        shared = self.shared_layer2(shared)
        
        # 单个维度处理
        dep_feat = self.dep_specific(shared)
        sui_feat = self.sui_specific(shared)
        risk_feat = self.risk_specific(shared)
        
        # 单个维度的回归输出
        dep_score = torch.sigmoid(self.dep_head(dep_feat)).squeeze(-1)
        sui_score = torch.sigmoid(self.sui_head(sui_feat)).squeeze(-1)
        risk_score_individual = torch.sigmoid(self.risk_head(risk_feat)).squeeze(-1)
        
        # 联合学习：融合所有特征
        fused_feat = torch.cat([dep_feat, sui_feat, risk_feat], dim=1)
        fused_feat = self.fusion_layer(fused_feat)
        risk_score_joint = torch.sigmoid(self.fusion_head(fused_feat)).squeeze(-1)
        
        # 最终风险评分（融合两个预测）
        risk_score = 0.6 * risk_score_individual + 0.4 * risk_score_joint
        
        if dep_target is not None:
            loss_dep = F.mse_loss(dep_score, dep_target)
            loss_sui = F.mse_loss(sui_score, sui_target)
            loss_risk_individual = F.mse_loss(risk_score_individual, risk_target)
            loss_risk_joint = F.mse_loss(risk_score_joint, risk_target)
            
            # 联合损失
            total_loss = (0.7 * loss_dep + 
                         1.2 * loss_sui + 
                         0.5 * loss_risk_individual + 
                         0.5 * loss_risk_joint)
            
            return {
                "loss": total_loss,
                "loss_dep": loss_dep,
                "loss_sui": loss_sui,
                "dep_score": dep_score,
                "sui_score": sui_score,
                "risk_score": risk_score
            }
        
        return {
            "dep_score": dep_score,
            "sui_score": sui_score,
            "risk_score": risk_score
        }

# ====================== 评估指标 ======================
def concordance_correlation_coefficient(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true)
    var_pred = np.var(y_pred)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    ccc = (2 * cov) / (var_true + var_pred + (mean_true - mean_pred) ** 2 + 1e-10)
    return ccc

# ====================== 训练函数 ======================
def train():
    print("\n[2/6] 加载预训练模型和分词器...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
    
    print("[3/6] 创建数据加载器...")
    train_dataset = JointRegressionDataset(train_texts, train_labels, tokenizer)
    val_dataset   = JointRegressionDataset(val_texts,   val_labels,   tokenizer)
    test_dataset  = JointRegressionDataset(test_texts,  test_labels,  tokenizer)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, drop_last=False)
    val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)
    
    print("[4/6] 初始化模型...")
    model = JointRegressionModel(MODEL_PATH).to(DEVICE)
    
    optimizer = AdamW(model.parameters(), lr=1.6e-5, weight_decay=1e-5)
    total_steps = len(train_loader) * 8
    scheduler = PolynomialLR(optimizer, total_iters=total_steps, power=1.0)
    
    best_val_loss = float('inf')
    best_state = None
    patience = 4
    patience_counter = 0
    
    print("\n[5/6] 开始训练...\n")
    for epoch in range(8):
        model.train()
        total_loss = 0
        total_loss_dep = 0
        total_loss_sui = 0
        
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} Train"):
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            dep_target     = batch["dep_target"].to(DEVICE)
            sui_target     = batch["sui_target"].to(DEVICE)
            risk_target    = batch["risk_target"].to(DEVICE)
            
            outputs = model(input_ids, attention_mask, dep_target, sui_target, risk_target)
            loss = outputs["loss"]
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
            total_loss_dep += outputs["loss_dep"].item()
            total_loss_sui += outputs["loss_sui"].item()
        
        avg_train_loss = total_loss / len(train_loader)
        avg_loss_dep = total_loss_dep / len(train_loader)
        avg_loss_sui = total_loss_sui / len(train_loader)
        
        print(f"Epoch {epoch+1} Train - Total: {avg_train_loss:.4f} | Dep: {avg_loss_dep:.4f} | Sui: {avg_loss_sui:.4f}")
        
        # 验证
        model.eval()
        val_dep_preds = []
        val_sui_preds = []
        val_risk_preds = []
        val_dep_targets = []
        val_sui_targets = []
        val_risk_targets = []
        val_labels_list = []
        val_loss = 0
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                dep_target     = batch["dep_target"].to(DEVICE)
                sui_target     = batch["sui_target"].to(DEVICE)
                risk_target    = batch["risk_target"].to(DEVICE)
                single_labels  = batch["single_label"].numpy()
                
                outputs = model(input_ids, attention_mask, dep_target, sui_target, risk_target)
                val_loss += outputs["loss"].item()
                
                val_dep_preds.extend(outputs["dep_score"].cpu().numpy())
                val_sui_preds.extend(outputs["sui_score"].cpu().numpy())
                val_risk_preds.extend(outputs["risk_score"].cpu().numpy())
                val_dep_targets.extend(dep_target.cpu().numpy())
                val_sui_targets.extend(sui_target.cpu().numpy())
                val_risk_targets.extend(risk_target.cpu().numpy())
                val_labels_list.extend(single_labels)
        
        avg_val_loss = val_loss / len(val_loader)
        val_dep_spear, _ = spearmanr(val_dep_targets, val_dep_preds)
        val_sui_spear, _ = spearmanr(val_sui_targets, val_sui_preds)
        val_risk_spear, _ = spearmanr(val_risk_targets, val_risk_preds)
        
        print(f"Epoch {epoch+1} Val - Loss: {avg_val_loss:.4f} | Dep: {val_dep_spear:.4f} | Sui: {val_sui_spear:.4f} | Risk: {val_risk_spear:.4f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_state = model.state_dict()
            torch.save(best_state, "best_joint_regression.pt")
            patience_counter = 0
            print(f"✓ Best model saved (Val Loss: {avg_val_loss:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    # 测试
    print("\n[6/6] 开始测试评估...\n")
    model.load_state_dict(torch.load("best_joint_regression.pt"))
    model.eval()
    
    test_dep_preds = []
    test_sui_preds = []
    test_risk_preds = []
    test_dep_targets = []
    test_sui_targets = []
    test_risk_targets = []
    test_labels_list = []
    
    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            dep_target     = batch["dep_target"].cpu().numpy()
            sui_target     = batch["sui_target"].cpu().numpy()
            risk_target    = batch["risk_target"].cpu().numpy()
            single_labels  = batch["single_label"].numpy()
            
            outputs = model(input_ids, attention_mask)
            
            test_dep_preds.extend(outputs["dep_score"].cpu().numpy())
            test_sui_preds.extend(outputs["sui_score"].cpu().numpy())
            test_risk_preds.extend(outputs["risk_score"].cpu().numpy())
            test_dep_targets.extend(dep_target)
            test_sui_targets.extend(sui_target)
            test_risk_targets.extend(risk_target)
            test_labels_list.extend(single_labels)
    
    test_dep_preds = np.array(test_dep_preds)
    test_sui_preds = np.array(test_sui_preds)
    test_risk_preds = np.array(test_risk_preds)
    test_dep_targets = np.array(test_dep_targets)
    test_sui_targets = np.array(test_sui_targets)
    test_risk_targets = np.array(test_risk_targets)
    test_labels_list = np.array(test_labels_list)
    
    # 计算指标
    dep_spear, dep_pvalue = spearmanr(test_dep_targets, test_dep_preds)
    dep_ccc = concordance_correlation_coefficient(test_dep_targets, test_dep_preds)
    dep_mae = np.mean(np.abs(test_dep_targets - test_dep_preds))
    dep_rmse = np.sqrt(np.mean((test_dep_targets - test_dep_preds) ** 2))
    
    sui_spear, sui_pvalue = spearmanr(test_sui_targets, test_sui_preds)
    sui_ccc = concordance_correlation_coefficient(test_sui_targets, test_sui_preds)
    sui_mae = np.mean(np.abs(test_sui_targets - test_sui_preds))
    sui_rmse = np.sqrt(np.mean((test_sui_targets - test_sui_preds) ** 2))
    
    risk_spear, risk_pvalue = spearmanr(test_risk_targets, test_risk_preds)
    risk_ccc = concordance_correlation_coefficient(test_risk_targets, test_risk_preds)
    risk_mae = np.mean(np.abs(test_risk_targets - test_risk_preds))
    risk_rmse = np.sqrt(np.mean((test_risk_targets - test_risk_preds) ** 2))
    
    print("\n" + "="*80)
    print("=== 联合回归模型 - 测试结果 ===")
    print("="*80)
    print(f"\nDepression Risk:")
    print(f"  Spearman: {dep_spear:.4f} (p-value: {dep_pvalue:.2e})")
    print(f"  CCC: {dep_ccc:.4f}")
    print(f"  MAE: {dep_mae:.4f}")
    print(f"  RMSE: {dep_rmse:.4f}")
    
    print(f"\nSuicidal Risk:")
    print(f"  Spearman: {sui_spear:.4f} (p-value: {sui_pvalue:.2e})")
    print(f"  CCC: {sui_ccc:.4f}")
    print(f"  MAE: {sui_mae:.4f}")
    print(f"  RMSE: {sui_rmse:.4f}")
    
    print(f"\nJoint Risk Score:")
    print(f"  Spearman: {risk_spear:.4f} (p-value: {risk_pvalue:.2e})")
    print(f"  CCC: {risk_ccc:.4f}")
    print(f"  MAE: {risk_mae:.4f}")
    print(f"  RMSE: {risk_rmse:.4f}")
    
    avg_spear = (dep_spear + sui_spear + risk_spear) / 3
    avg_ccc = (dep_ccc + sui_ccc + risk_ccc) / 3
    print(f"\n平均 Spearman: {avg_spear:.4f}")
    print(f"平均 CCC: {avg_ccc:.4f}")
    
    # 高风险检测
    high_risk_true = (test_labels_list == 3).astype(int)
    threshold = 0.72
    high_risk_pred = (test_risk_preds >= threshold).astype(int)
    
    auc = roc_auc_score(high_risk_true, test_risk_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(high_risk_true, high_risk_pred, average='binary')
    
    print("\n" + "="*80)
    print(f"=== 高风险检测 (threshold = {threshold}) ===")
    print("="*80)
    print(f"AUC: {auc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    
    # 可视化
    print("\n生成可视化图表...")
    
    plt.figure(figsize=(12, 9))
    colors = ['#3498db', '#e74c3c', '#95a5a6', '#c0392b']
    class_names = label_encoder.classes_
    
    for i in range(4):
        mask = test_labels_list == i
        plt.scatter(test_dep_preds[mask], 
                    test_sui_preds[mask], 
                    c=colors[i], label=f'{class_names[i]} (n={mask.sum()})', 
                    alpha=0.7, s=50, edgecolors='black', linewidth=0.5)
    
    plt.axvline(x=0.60, color='orange', linestyle='--', alpha=0.5, linewidth=2)
    plt.axhline(y=threshold, color='red', linestyle='--', alpha=0.5, linewidth=2)
    
    plt.xlabel('Predicted Depression Severity', fontsize=12, fontweight='bold')
    plt.ylabel('Predicted Suicidal Risk', fontsize=12, fontweight='bold')
    plt.title('2D Risk Space - Joint Regression', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10, loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.xlim(-0.05, 1.05)
    plt.ylim(-0.05, 1.05)
    plt.tight_layout()
    plt.savefig('3_2d_risk_space.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ 2D risk space 已保存为 3_2d_risk_space.png")
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].scatter(test_dep_targets, test_dep_preds, alpha=0.6, s=30, c='#3498db', edgecolors='black', linewidth=0.5)
    axes[0].plot([0, 1], [0, 1], 'r--', lw=2)
    axes[0].set_xlabel('True Target', fontweight='bold')
    axes[0].set_ylabel('Predicted Score', fontweight='bold')
    axes[0].set_title(f'Depression\nSpearman={dep_spear:.4f}, CCC={dep_ccc:.4f}', fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim(-0.05, 1.05)
    axes[0].set_ylim(-0.05, 1.05)
    
    axes[1].scatter(test_sui_targets, test_sui_preds, alpha=0.6, s=30, c='#e74c3c', edgecolors='black', linewidth=0.5)
    axes[1].plot([0, 1], [0, 1], 'r--', lw=2)
    axes[1].set_xlabel('True Target', fontweight='bold')
    axes[1].set_ylabel('Predicted Score', fontweight='bold')
    axes[1].set_title(f'Suicidal Risk\nSpearman={sui_spear:.4f}, CCC={sui_ccc:.4f}', fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim(-0.05, 1.05)
    axes[1].set_ylim(-0.05, 1.05)
    
    axes[2].scatter(test_risk_targets, test_risk_preds, alpha=0.6, s=30, c='#27ae60', edgecolors='black', linewidth=0.5)
    axes[2].plot([0, 1], [0, 1], 'r--', lw=2)
    axes[2].set_xlabel('True Target', fontweight='bold')
    axes[2].set_ylabel('Predicted Score', fontweight='bold')
    axes[2].set_title(f'Joint Risk\nSpearman={risk_spear:.4f}, CCC={risk_ccc:.4f}', fontweight='bold')
    axes[2].grid(True, alpha=0.3)
    axes[2].set_xlim(-0.05, 1.05)
    axes[2].set_ylim(-0.05, 1.05)
    
    plt.tight_layout()
    plt.savefig('3_prediction_vs_target.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ 预测 vs 真实值已保存为 3_prediction_vs_target.png")
    
    df_violin = pd.DataFrame({
        'Depression Score': test_dep_preds,
        'Suicidal Risk Score': test_sui_preds,
        'Joint Risk Score': test_risk_preds,
        'True Class': [class_names[l] for l in test_labels_list]
    })
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    sns.violinplot(data=df_violin, x='True Class', y='Depression Score', ax=axes[0], palette='Set2')
    axes[0].set_title('Depression Score Distribution', fontweight='bold')
    axes[0].set_ylabel('Predicted Score (0-1)')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    sns.violinplot(data=df_violin, x='True Class', y='Suicidal Risk Score', ax=axes[1], palette='Set2')
    axes[1].set_title('Suicidal Risk Score Distribution', fontweight='bold')
    axes[1].set_ylabel('Predicted Score (0-1)')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    sns.violinplot(data=df_violin, x='True Class', y='Joint Risk Score', ax=axes[2], palette='Set2')
    axes[2].set_title('Joint Risk Score Distribution', fontweight='bold')
    axes[2].set_ylabel('Predicted Score (0-1)')
    axes[2].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('3_score_distribution.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ 分布图已保存为 3_score_distribution.png")
    
    print("\n✓ 联合回归模型实验完成！\n")

if __name__ == "__main__":
    train()


Using device: cuda

[1/6] 加载数据...
数据集大小: 49612
类别: ['Anxiety' 'Depression' 'Normal' 'Suicidal']
训练集: 34728, 验证集: 7442, 测试集: 7442

[2/6] 加载预训练模型和分词器...
[3/6] 创建数据加载器...
[4/6] 初始化模型...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: ./hf_models\roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



[5/6] 开始训练...



Epoch 1 Train: 100%|█████████████████████████████████████████████████████████████| 2171/2171 [06:45<00:00,  5.35it/s]


...

Epoch 6 Train - Total: 0.0222 | Dep: 0.0045 | Sui: 0.0110
Epoch 6 Val - Loss: 0.0607 | Dep: 0.9082 | Sui: 0.9076 | Risk: 0.9081


Epoch 7 Train: 100%|█████████████████████████████████████████████████████████████| 2171/2171 [09:05<00:00,  3.98it/s]


Epoch 7 Train - Total: 0.0177 | Dep: 0.0035 | Sui: 0.0089
Epoch 7 Val - Loss: 0.0612 | Dep: 0.9077 | Sui: 0.9080 | Risk: 0.9081


Epoch 8 Train: 100%|█████████████████████████████████████████████████████████████| 2171/2171 [09:01<00:00,  4.01it/s]


Epoch 8 Train - Total: 0.0151 | Dep: 0.0031 | Sui: 0.0075
Epoch 8 Val - Loss: 0.0638 | Dep: 0.9062 | Sui: 0.9062 | Risk: 0.9064
Early stopping at epoch 8

[6/6] 开始测试评估...


=== 联合回归模型 - 测试结果 ===

Depression Risk:
  Spearman: 0.9062 (p-value: 0.00e+00)
  CCC: 0.9374
  MAE: 0.0492
  RMSE: 0.1273

Suicidal Risk:
  Spearman: 0.9059 (p-value: 0.00e+00)
  CCC: 0.8873
  MAE: 0.0763
  RMSE: 0.1676

Joint Risk Score:
  Spearman: 0.9062 (p-value: 0.00e+00)
  CCC: 0.9304
  MAE: 0.0585
  RMSE: 0.1381

平均 Spearman: 0.9061
平均 CCC: 0.9183

=== 高风险检测 (threshold = 0.72) ===
AUC: 0.9495
Precision: 0.6511
Recall: 0.9340
F1-Score: 0.7673

生成可视化图表...
✓ 2D risk space 已保存为 3_2d_risk_space.png
✓ 预测 vs 真实值已保存为 3_prediction_vs_target.png
✓ 分布图已保存为 3_score_distribution.png

✓ 联合回归模型实验完成！



In [ ]:
#模型 4：层级回归模型（Hierarchical Regression）

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from torch.optim.lr_scheduler import PolynomialLR
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder
from scipy.stats import spearmanr
import os
import random
from tqdm import tqdm
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

LOCAL_MODEL_DIR = "./hf_models"
MODEL_PATH = os.path.join(LOCAL_MODEL_DIR, "roberta-base")

# ====================== 数据加载 ======================
print("\n[1/6] 加载数据...")
df = pd.read_csv("mental_heath_unbanlanced.csv")
df = df.dropna(subset=["text", "status"])
df["text"] = df["text"].astype(str)
print(f"数据集大小: {len(df)}")

label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df["status"])
print(f"类别: {label_encoder.classes_}")

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df["text"].tolist(),
    df["label_encoded"].tolist(),
    test_size=0.3,
    random_state=42,
    stratify=df["label_encoded"]
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

print(f"训练集: {len(train_texts)}, 验证集: {len(val_texts)}, 测试集: {len(test_texts)}")

# ====================== 回归目标定义 ======================
def create_regression_targets(single_labels):
    """
    层级回归：粗粒度 -> 中粒度 -> 细粒度
    """
    batch_size = len(single_labels)
    dep_target = torch.zeros(batch_size, dtype=torch.float32)
    sui_target = torch.zeros(batch_size, dtype=torch.float32)
    
    for i, lab in enumerate(single_labels):
        if lab == 2:   # Normal
            dep_target[i] = 0.10
            sui_target[i] = 0.03
        elif lab == 0: # Anxiety
            dep_target[i] = 0.35
            sui_target[i] = 0.15
        elif lab == 1: # Depression
            dep_target[i] = 0.80
            sui_target[i] = 0.45
        elif lab == 3: # Suicidal
            dep_target[i] = 0.92
            sui_target[i] = 0.95
    
    return dep_target, sui_target

# ====================== Dataset ======================
class HierarchicalRegressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=96):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        single_label = self.labels[idx]
        dep_target, sui_target = create_regression_targets([single_label])
        
        item = {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "dep_target": dep_target.squeeze(0),
            "sui_target": sui_target.squeeze(0),
            "single_label": torch.tensor(single_label, dtype=torch.long),
        }
        return item

# ====================== 层级回归模型 ======================
class HierarchicalRegressionModel(nn.Module):
    """
    层级回归模型
    通过多层递进式回归头逐步精化预测
    使用跳跃连接和多层监督
    """
    def __init__(self, model_path):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_path, local_files_only=True)
        hidden = self.encoder.config.hidden_size
        
        # 特征融合
        self.dropout_main = nn.Dropout(0.2)
        self.layer_norm = nn.LayerNorm(hidden)
        self.attention_weights = nn.Linear(hidden, 1)
        
        # ========== 第1层：粗粒度 ==========
        self.layer1 = nn.Sequential(
            nn.Linear(hidden, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        
        self.dep_head1 = nn.Linear(512, 1)
        self.sui_head1 = nn.Linear(512, 1)
        
        # ========== 第2层：中粒度 ==========
        self.layer2 = nn.Sequential(
            nn.Linear(512 + 2, 256),  # +2 for previous predictions
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.15)
        )
        
        self.dep_head2 = nn.Linear(256, 1)
        self.sui_head2 = nn.Linear(256, 1)
        
        # ========== 第3层：细粒度 ==========
        self.layer3 = nn.Sequential(
            nn.Linear(256 + 2, 128),  # +2 for previous predictions
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1)
        )
        
        self.dep_head3 = nn.Linear(128, 1)
        self.sui_head3 = nn.Linear(128, 1)

    def forward(self, input_ids, attention_mask, dep_target=None, sui_target=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        
        # 特征融合
        cls_token = last_hidden[:, 0, :]
        attention_logits = self.attention_weights(last_hidden)
        attention_weights = F.softmax(attention_logits, dim=1)
        attention_pooled = torch.sum(last_hidden * attention_weights, dim=1)
        
        mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        sum_hidden = torch.sum(last_hidden * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        mean_pooled = sum_hidden / sum_mask
        
        fused = cls_token + 0.4 * attention_pooled + 0.3 * mean_pooled
        fused = self.layer_norm(fused)
        fused = self.dropout_main(fused)
        
        # ========== 第1层预测 ==========
        feat1 = self.layer1(fused)
        dep_logit1 = self.dep_head1(feat1)
        sui_logit1 = self.sui_head1(feat1)
        dep_score1 = torch.sigmoid(dep_logit1)
        sui_score1 = torch.sigmoid(sui_logit1)
        
        # ========== 第2层预测 ==========
        feat2_input = torch.cat([feat1, dep_score1, sui_score1], dim=1)
        feat2 = self.layer2(feat2_input)
        dep_logit2 = self.dep_head2(feat2)
        sui_logit2 = self.sui_head2(feat2)
        dep_score2 = torch.sigmoid(dep_logit2)
        sui_score2 = torch.sigmoid(sui_logit2)
        
        # ========== 第3层预测（最终） ==========
        feat3_input = torch.cat([feat2, dep_score2, sui_score2], dim=1)
        feat3 = self.layer3(feat3_input)
        dep_logit3 = self.dep_head3(feat3)
        sui_logit3 = self.sui_head3(feat3)
        dep_score = torch.sigmoid(dep_logit3).squeeze(-1)
        sui_score = torch.sigmoid(sui_logit3).squeeze(-1)
        
        if dep_target is not None:
            # 多层监督
            loss_dep1 = F.mse_loss(dep_score1.squeeze(-1), dep_target)
            loss_sui1 = F.mse_loss(sui_score1.squeeze(-1), sui_target)
            
            loss_dep2 = F.mse_loss(dep_score2.squeeze(-1), dep_target)
            loss_sui2 = F.mse_loss(sui_score2.squeeze(-1), sui_target)
            
            loss_dep3 = F.mse_loss(dep_score, dep_target)
            loss_sui3 = F.mse_loss(sui_score, sui_target)
            
            # 加权组合：最后一层权重最高
            total_loss = (0.2 * (loss_dep1 + 1.3 * loss_sui1) +
                         0.4 * (loss_dep2 + 1.3 * loss_sui2) +
                         1.0 * (loss_dep3 + 1.3 * loss_sui3))
            
            return {
                "loss": total_loss,
                "loss_dep": loss_dep3,
                "loss_sui": loss_sui3,
                "dep_score": dep_score,
                "sui_score": sui_score
            }
        
        return {
            "dep_score": dep_score,
            "sui_score": sui_score
        }

# ====================== 评估指标 ======================
def concordance_correlation_coefficient(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true)
    var_pred = np.var(y_pred)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    ccc = (2 * cov) / (var_true + var_pred + (mean_true - mean_pred) ** 2 + 1e-10)
    return ccc

# ====================== 训练函数 ======================
def train():
    print("\n[2/6] 加载预训练模型和分词器...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
    
    print("[3/6] 创建数据加载器...")
    train_dataset = HierarchicalRegressionDataset(train_texts, train_labels, tokenizer)
    val_dataset   = HierarchicalRegressionDataset(val_texts,   val_labels,   tokenizer)
    test_dataset  = HierarchicalRegressionDataset(test_texts,  test_labels,  tokenizer)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, drop_last=False)
    val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)
    
    print("[4/6] 初始化模型...")
    model = HierarchicalRegressionModel(MODEL_PATH).to(DEVICE)
    
    optimizer = AdamW(model.parameters(), lr=1.6e-5, weight_decay=1e-5)
    total_steps = len(train_loader) * 8
    scheduler = PolynomialLR(optimizer, total_iters=total_steps, power=1.0)
    
    best_val_loss = float('inf')
    best_state = None
    patience = 4
    patience_counter = 0
    
    print("\n[5/6] 开始训练...\n")
    for epoch in range(8):
        model.train()
        total_loss = 0
        total_loss_dep = 0
        total_loss_sui = 0
        
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} Train"):
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            dep_target     = batch["dep_target"].to(DEVICE)
            sui_target     = batch["sui_target"].to(DEVICE)
            
            outputs = model(input_ids, attention_mask, dep_target, sui_target)
            loss = outputs["loss"]
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
            total_loss_dep += outputs["loss_dep"].item()
            total_loss_sui += outputs["loss_sui"].item()
        
        avg_train_loss = total_loss / len(train_loader)
        avg_loss_dep = total_loss_dep / len(train_loader)
        avg_loss_sui = total_loss_sui / len(train_loader)
        
        print(f"Epoch {epoch+1} Train - Total: {avg_train_loss:.4f} | Dep: {avg_loss_dep:.4f} | Sui: {avg_loss_sui:.4f}")
        
        # 验证
        model.eval()
        val_dep_preds = []
        val_sui_preds = []
        val_dep_targets = []
        val_sui_targets = []
        val_labels_list = []
        val_loss = 0
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                dep_target     = batch["dep_target"].to(DEVICE)
                sui_target     = batch["sui_target"].to(DEVICE)
                single_labels  = batch["single_label"].numpy()
                
                outputs = model(input_ids, attention_mask, dep_target, sui_target)
                val_loss += outputs["loss"].item()
                
                val_dep_preds.extend(outputs["dep_score"].cpu().numpy())
                val_sui_preds.extend(outputs["sui_score"].cpu().numpy())
                val_dep_targets.extend(dep_target.cpu().numpy())
                val_sui_targets.extend(sui_target.cpu().numpy())
                val_labels_list.extend(single_labels)
        
        avg_val_loss = val_loss / len(val_loader)
        val_dep_spear, _ = spearmanr(val_dep_targets, val_dep_preds)
        val_sui_spear, _ = spearmanr(val_sui_targets, val_sui_preds)
        
        print(f"Epoch {epoch+1} Val - Loss: {avg_val_loss:.4f} | Dep: {val_dep_spear:.4f} | Sui: {val_sui_spear:.4f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_state = model.state_dict()
            torch.save(best_state, "best_hierarchical_regression.pt")
            patience_counter = 0
            print(f"✓ Best model saved (Val Loss: {avg_val_loss:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    # 测试
    print("\n[6/6] 开始测试评估...\n")
    model.load_state_dict(torch.load("best_hierarchical_regression.pt"))
    model.eval()
    
    test_dep_preds = []
    test_sui_preds = []
    test_dep_targets = []
    test_sui_targets = []
    test_labels_list = []
    
    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            dep_target     = batch["dep_target"].cpu().numpy()
            sui_target     = batch["sui_target"].cpu().numpy()
            single_labels  = batch["single_label"].numpy()
            
            outputs = model(input_ids, attention_mask)
            
            test_dep_preds.extend(outputs["dep_score"].cpu().numpy())
            test_sui_preds.extend(outputs["sui_score"].cpu().numpy())
            test_dep_targets.extend(dep_target)
            test_sui_targets.extend(sui_target)
            test_labels_list.extend(single_labels)
    
    test_dep_preds = np.array(test_dep_preds)
    test_sui_preds = np.array(test_sui_preds)
    test_dep_targets = np.array(test_dep_targets)
    test_sui_targets = np.array(test_sui_targets)
    test_labels_list = np.array(test_labels_list)
    
    # 计算指标
    dep_spear, dep_pvalue = spearmanr(test_dep_targets, test_dep_preds)
    dep_ccc = concordance_correlation_coefficient(test_dep_targets, test_dep_preds)
    dep_mae = np.mean(np.abs(test_dep_targets - test_dep_preds))
    dep_rmse = np.sqrt(np.mean((test_dep_targets - test_dep_preds) ** 2))
    
    sui_spear, sui_pvalue = spearmanr(test_sui_targets, test_sui_preds)
    sui_ccc = concordance_correlation_coefficient(test_sui_targets, test_sui_preds)
    sui_mae = np.mean(np.abs(test_sui_targets - test_sui_preds))
    sui_rmse = np.sqrt(np.mean((test_sui_targets - test_sui_preds) ** 2))
    
    print("\n" + "="*80)
    print("=== 层级回归模型 - 测试结果 ===")
    print("="*80)
    print(f"\nDepression Risk:")
    print(f"  Spearman: {dep_spear:.4f} (p-value: {dep_pvalue:.2e})")
    print(f"  CCC: {dep_ccc:.4f}")
    print(f"  MAE: {dep_mae:.4f}")
    print(f"  RMSE: {dep_rmse:.4f}")
    
    print(f"\nSuicidal Risk:")
    print(f"  Spearman: {sui_spear:.4f} (p-value: {sui_pvalue:.2e})")
    print(f"  CCC: {sui_ccc:.4f}")
    print(f"  MAE: {sui_mae:.4f}")
    print(f"  RMSE: {sui_rmse:.4f}")
    
    avg_spear = (dep_spear + sui_spear) / 2
    avg_ccc = (dep_ccc + sui_ccc) / 2
    print(f"\n平均 Spearman: {avg_spear:.4f}")
    print(f"平均 CCC: {avg_ccc:.4f}")
    
    # 高风险检测
    high_risk_true = (test_labels_list == 3).astype(int)
    threshold = 0.72
    high_risk_pred = (test_sui_preds >= threshold).astype(int)
    
    auc = roc_auc_score(high_risk_true, test_sui_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(high_risk_true, high_risk_pred, average='binary')
    
    print("\n" + "="*80)
    print(f"=== 高风险检测 (threshold = {threshold}) ===")
    print("="*80)
    print(f"AUC: {auc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    
    # 可视化
    print("\n生成可视化图表...")
    
    plt.figure(figsize=(12, 9))
    colors = ['#3498db', '#e74c3c', '#95a5a6', '#c0392b']
    class_names = label_encoder.classes_
    
    for i in range(4):
        mask = test_labels_list == i
        plt.scatter(test_dep_preds[mask], 
                    test_sui_preds[mask], 
                    c=colors[i], label=f'{class_names[i]} (n={mask.sum()})', 
                    alpha=0.7, s=50, edgecolors='black', linewidth=0.5)
    
    plt.axvline(x=0.60, color='orange', linestyle='--', alpha=0.5, linewidth=2)
    plt.axhline(y=threshold, color='red', linestyle='--', alpha=0.5, linewidth=2)
    
    plt.xlabel('Predicted Depression Severity', fontsize=12, fontweight='bold')
    plt.ylabel('Predicted Suicidal Risk', fontsize=12, fontweight='bold')
    plt.title('2D Risk Space - Hierarchical Regression', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10, loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.xlim(-0.05, 1.05)
    plt.ylim(-0.05, 1.05)
    plt.tight_layout()
    plt.savefig('4_2d_risk_space.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ 2D risk space 已保存为 4_2d_risk_space.png")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].scatter(test_dep_targets, test_dep_preds, alpha=0.6, s=30, c='#3498db', edgecolors='black', linewidth=0.5)
    axes[0].plot([0, 1], [0, 1], 'r--', lw=2)
    axes[0].set_xlabel('True Target', fontweight='bold')
    axes[0].set_ylabel('Predicted Score', fontweight='bold')
    axes[0].set_title(f'Depression\nSpearman={dep_spear:.4f}, CCC={dep_ccc:.4f}', fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim(-0.05, 1.05)
    axes[0].set_ylim(-0.05, 1.05)
    
    axes[1].scatter(test_sui_targets, test_sui_preds, alpha=0.6, s=30, c='#e74c3c', edgecolors='black', linewidth=0.5)
    axes[1].plot([0, 1], [0, 1], 'r--', lw=2)
    axes[1].set_xlabel('True Target', fontweight='bold')
    axes[1].set_ylabel('Predicted Score', fontweight='bold')
    axes[1].set_title(f'Suicidal Risk\nSpearman={sui_spear:.4f}, CCC={sui_ccc:.4f}', fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim(-0.05, 1.05)
    axes[1].set_ylim(-0.05, 1.05)
    
    plt.tight_layout()
    plt.savefig('4_prediction_vs_target.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ 预测 vs 真实值已保存为 4_prediction_vs_target.png")
    
    df_violin = pd.DataFrame({
        'Depression Score': test_dep_preds,
        'Suicidal Risk Score': test_sui_preds,
        'True Class': [class_names[l] for l in test_labels_list]
    })
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    sns.violinplot(data=df_violin, x='True Class', y='Depression Score', ax=axes[0], palette='Set2')
    axes[0].set_title('Depression Score Distribution', fontweight='bold')
    axes[0].set_ylabel('Predicted Score (0-1)')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    sns.violinplot(data=df_violin, x='True Class', y='Suicidal Risk Score', ax=axes[1], palette='Set2')
    axes[1].set_title('Suicidal Risk Score Distribution', fontweight='bold')
    axes[1].set_ylabel('Predicted Score (0-1)')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('4_score_distribution.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ 分布图已保存为 4_score_distribution.png")
    
    print("\n✓ 层级回归模型实验完成！\n")

if __name__ == "__main__":
    train()


Using device: cuda

[1/6] 加载数据...
数据集大小: 49612
类别: ['Anxiety' 'Depression' 'Normal' 'Suicidal']
训练集: 34728, 验证集: 7442, 测试集: 7442

[2/6] 加载预训练模型和分词器...
[3/6] 创建数据加载器...
[4/6] 初始化模型...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: ./hf_models\roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



[5/6] 开始训练...



Epoch 1 Train: 100%|█████████████████████████████████████████████████████████████| 2171/2171 [09:12<00:00,  3.93it/s]


...

Epoch 6 Train - Total: 0.0294 | Dep: 0.0046 | Sui: 0.0106
Epoch 6 Val - Loss: 0.0815 | Dep: 0.9060 | Sui: 0.9045


Epoch 7 Train: 100%|█████████████████████████████████████████████████████████████| 2171/2171 [09:11<00:00,  3.94it/s]


Epoch 7 Train - Total: 0.0233 | Dep: 0.0035 | Sui: 0.0084
Epoch 7 Val - Loss: 0.0812 | Dep: 0.9077 | Sui: 0.9075


Epoch 8 Train: 100%|█████████████████████████████████████████████████████████████| 2171/2171 [08:59<00:00,  4.02it/s]


Epoch 8 Train - Total: 0.0197 | Dep: 0.0030 | Sui: 0.0071
Epoch 8 Val - Loss: 0.0842 | Dep: 0.9075 | Sui: 0.9066
Early stopping at epoch 8

[6/6] 开始测试评估...


=== 层级回归模型 - 测试结果 ===

Depression Risk:
  Spearman: 0.9086 (p-value: 0.00e+00)
  CCC: 0.9408
  MAE: 0.0489
  RMSE: 0.1242

Suicidal Risk:
  Spearman: 0.9072 (p-value: 0.00e+00)
  CCC: 0.8948
  MAE: 0.0798
  RMSE: 0.1610

平均 Spearman: 0.9079
平均 CCC: 0.9178

=== 高风险检测 (threshold = 0.72) ===
AUC: 0.9523
Precision: 0.7452
Recall: 0.8365
F1-Score: 0.7882

生成可视化图表...
✓ 2D risk space 已保存为 4_2d_risk_space.png
✓ 预测 vs 真实值已保存为 4_prediction_vs_target.png
✓ 分布图已保存为 4_score_distribution.png

✓ 层级回归模型实验完成！

